# Environment configuration — landing/hrm
Credentials are external. Set PIPELINE_ENV and job variables generated from config/environments and config/secrets.
This repository revision changes configuration handling, not the source ingestion/write algorithms.


# NB_landing.hrm.employee — batch upsert theo ngày
Giữ cấu hình kết nối S3/metastore/Spark từ notebook mẫu. Không cần các cờ confirm.
API trả các bản ghi upsert trong khoảng request: date_start=ETL_Date, date_end=ETL_Date+1 ngày.
Mặc định ETL_Date=T−1 theo giờ Việt Nam; data_date giữ YYYY-MM-DD.
Landing: `s3://landing-bucket/hrm/employee/YYYYMMDD/page-000001.json` và `manifest.json`.
Mỗi ngày chỉ có một tập file hiện hành, không có thư mục runs. RUN_ID vẫn lưu trong manifest/log để truy vết.


In [ ]:
# API -> Landing không khởi tạo Spark; không cần spark.stop().
# Toàn bộ cấu hình và hàm init_spark của mẫu vẫn được giữ ở các cell phía dưới.


In [ ]:
import logging, time, os, json, ast, traceback, re, uuid, hashlib, sys
from datetime import datetime, timedelta, date, timezone
from zoneinfo import ZoneInfo
from urllib.parse import urlparse
import boto3
from botocore.config import Config
import requests


In [ ]:
# Log Setting
logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
    force=True 
)
logger = logging.getLogger(__name__)

In [ ]:
# start_date = datetime.now() + timedelta(hours=7)
start_date = datetime.now()
log_message = None
status = "Success"
record_count = 0
df = None

In [ ]:
# Environment configuration is injected by CP4D/job environment variables.
# Generate a job env file with scripts/build_job_env.py. Never embed credentials here.
def _need_env(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError("Thiếu biến môi trường: " + name)
    return value

PIPELINE_ENV = _need_env("PIPELINE_ENV")
if PIPELINE_ENV not in ("dev", "uat", "prod"):
    raise ValueError("PIPELINE_ENV phải là dev, uat hoặc prod")

# 1. Lấy các biến môi trường
now = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh"))
yesterday = now - timedelta(days=1)

# 3. Định dạng lại thành chuỗi YYYYMMDD
yesterday = yesterday.strftime("%Y-%m-%d")

#S3
S3_ENDPOINT = _need_env("S3_ENDPOINT")
ACCESS_KEY = _need_env("ACCESS_KEY")
SECRET_KEY = _need_env("SECRET_KEY")
BUCKET_NAME = _need_env("BUCKET_NAME")

#Metastore
WXD_HMS_USERNAME = os.getenv("WXD_HMS_USERNAME", "")
WXD_HMS_PASSWORD = os.getenv("WXD_HMS_PASSWORD", "")
ICEBERG_CATALOG1 = os.getenv("ICEBERG_CATALOG1", "bronze")
BUCKET_NAME1 = os.getenv("BUCKET_NAME1", "")
ACCESS_KEY1 = os.getenv("ACCESS_KEY1") or ACCESS_KEY
SECRET_KEY1 = os.getenv("SECRET_KEY1") or SECRET_KEY

HIVEMETASTORE_HOST = os.getenv("HIVEMETASTORE_HOST", "")

HIVE_TRUSTSTORE_PATH = os.getenv("HIVE_TRUSTSTORE_PATH", "")
HIVE_TRUSTSTORE_PASSWORD = os.getenv("HIVE_TRUSTSTORE_PASSWORD", "")

WXD_INSTANCE_ID = os.getenv("WXD_INSTANCE_ID", "")
HADOOP_AMS_URL = os.getenv("HADOOP_AMS_URL", "")
HADOOP_CAS_ENDPOINT = os.getenv("HADOOP_CAS_ENDPOINT", "")
HADOOP_CPG_ENDPOINT = os.getenv("HADOOP_CPG_ENDPOINT", "")


TABLE_NAME=os.getenv("TABLE_NAME","bronze.hrm.employee")
SOURCE_NAME=os.getenv("SOURCE_NAME","hrm.api.employee")
STEP='api_2_landing'


ETL_Date = os.getenv("ETL_Date",yesterday)

# HRM: entity và tham số chạy
ENTITY = 'employee'
LAYER = 'landing'
LOG_TABLE = os.getenv("LOG_TABLE", "bronze.pipeline.log_pipeline")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", ICEBERG_CATALOG1):
    raise ValueError("ICEBERG_CATALOG1 không hợp lệ")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", TABLE_NAME):
    raise ValueError("TABLE_NAME phải là catalog.schema.table hợp lệ")
# Truyền ETL_Date qua job hoặc sửa dòng sau trước khi chạy các cell tiếp theo:
# ETL_Date = "2026-10-20"

# Cấu hình API: môi trường job được ưu tiên so với giá trị trong notebook.
# HRM URL/token are supplied by the selected environment profile.
_need_env("HRM_BASE_URL")
_need_env("HRM_TOKEN")
ca = os.getenv("HRM_CA_BUNDLE") or True

# API thực tế đã kiểm chứng: luôn gửi Bearer header và access_token trong JSON.


In [ ]:
def init_spark():
    spark = SparkSession.builder \
        .appName(f"import_table") \
        .config("spark.sql.extensions","org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}","org.apache.iceberg.spark.SparkCatalog") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.type", "hive") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.uri",
                f"{HIVEMETASTORE_HOST}?catalog={ICEBERG_CATALOG1}") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.warehouse",
                f"s3a://{BUCKET_NAME1}") \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.hive.metastore.client.plain.username",
                WXD_HMS_USERNAME) \
        .config(f"spark.sql.catalog.{ICEBERG_CATALOG1}.hive.metastore.client.plain.password",
                WXD_HMS_PASSWORD) \
        .config("spark.hive.metastore.uris",HIVEMETASTORE_HOST) \
        .config("spark.hive.metastore.use.SSL", "true") \
        .config("spark.hive.metastore.truststore.type", "JKS") \
        .config("spark.hive.metastore.truststore.path", HIVE_TRUSTSTORE_PATH) \
        .config("spark.hive.metastore.truststore.password", HIVE_TRUSTSTORE_PASSWORD) \
        .config("spark.hive.metastore.client.auth.mode", "PLAIN") \
        .config("spark.hive.metastore.client.plain.username", WXD_HMS_USERNAME) \
        .config("spark.hive.metastore.client.plain.password", WXD_HMS_PASSWORD) \
        .config("spark.hive.metastore.client.thrift.transport.mode", "http") \
        .config("spark.hadoop.wxd.ams.url", HADOOP_AMS_URL) \
        .config("spark.hadoop.wxd.cas.endpoint", HADOOP_CAS_ENDPOINT) \
        .config("spark.hadoop.wxd.cpg.endpoint", HADOOP_CPG_ENDPOINT) \
        .config("spark.hadoop.wxd.instanceId", WXD_INSTANCE_ID) \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        \
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT) \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") \
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY) \
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY) \
        .config("spark.hadoop.fs.s3a.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.endpoint", S3_ENDPOINT) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.access.key", ACCESS_KEY) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.secret.key", SECRET_KEY) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.path.style.access", "true") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME}.connection.ssl.enabled", "true") \
        \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.endpoint", S3_ENDPOINT) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.access.key", ACCESS_KEY1) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.secret.key", SECRET_KEY1) \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.path.style.access", "true") \
        .config(f"spark.hadoop.fs.s3a.bucket.{BUCKET_NAME1}.connection.ssl.enabled", "true") \
        \
        .config("spark.hadoop.iceberg.engine.hive.lock-enabled", "false") \
        .config("spark.sql.catalogImplementation", "hive") \
        .getOrCreate()


    spark.sparkContext.setLogLevel("WARN")
    return spark

In [ ]:
def required(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError("Thiếu cấu hình: " + name)
    return value

BATCH_DATE = date.fromisoformat(ETL_Date)
if BATCH_DATE.isoformat() != ETL_Date:
    raise ValueError("ETL_Date phải đúng YYYY-MM-DD")
RUN_ID = uuid.uuid4().hex
ALLOW_EMPTY = True  # API success với 0 upsert là batch rỗng hợp lệ.
DAY_PREFIX = f"hrm/{ENTITY}/{BATCH_DATE.strftime('%Y%m%d')}"
LOG_PREFIX = f"hrm/{ENTITY}/_logs/{BATCH_DATE.strftime('%Y%m%d')}"
SOURCE = SOURCE_NAME

def pack(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")

def prefix(run_id):
    if not re.fullmatch(r"[0-9a-f]{32}", run_id):
        raise ValueError("run_id không hợp lệ")
    return DAY_PREFIX

def init_landing():
    # Reuse exactly S3_ENDPOINT, ACCESS_KEY, SECRET_KEY, BUCKET_NAME from template.
    if not SECRET_KEY or SECRET_KEY.startswith("{dsnextenc}"):
        raise ValueError("SECRET_KEY phải là secret S3 dạng gốc như file mẫu")
    return boto3.client("s3", endpoint_url=S3_ENDPOINT,
        aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY,
        config=Config(signature_version="s3v4", connect_timeout=15, read_timeout=60,
                      retries={"max_attempts": 3, "mode": "standard"},
                      s3={"addressing_style": "path"}))

def put_json(key, obj):
    blob = pack(obj)
    s3.put_object(Bucket=BUCKET_NAME, Key=key, Body=blob, ContentType="application/json")
    return hashlib.sha256(blob).hexdigest()

def get_json(key, digest=None):
    blob = s3.get_object(Bucket=BUCKET_NAME, Key=key)["Body"].read()
    if digest and hashlib.sha256(blob).hexdigest() != digest:
        raise ValueError("Landing checksum không khớp")
    return json.loads(blob)

def error_summary(exc):
    # Các ValueError do notebook tạo không chứa token hay nội dung nhân sự.
    if isinstance(exc, ValueError) and not isinstance(exc, json.JSONDecodeError):
        return str(exc)
    response = getattr(exc, "response", None)
    if isinstance(response, dict):
        return "S3 error: " + str(response.get("Error", {}).get("Code", "unknown"))
    return type(exc).__name__ + ": kiểm tra kết nối/runtime; không ghi response hoặc credentials."


In [ ]:
def validate_contract():
    required("HRM_BASE_URL")
    required("HRM_TOKEN")
    # Không dùng HRM_AUTH_MODE: request luôn có cả Bearer header và JSON token.

def clear_daily_landing():
    # Chỉ xóa prefix của đúng nguồn/entity/ngày, luôn có dấu / cuối.
    expected = f"hrm/{ENTITY}/{BATCH_DATE.strftime('%Y%m%d')}/"
    if DAY_PREFIX + "/" != expected or ENTITY not in ("unit", "employee"):
        raise ValueError("Từ chối xóa: prefix ngày không hợp lệ")
    keys = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET_NAME, Prefix=expected):
        keys.extend(item["Key"] for item in page.get("Contents", []))
    # Gỡ manifest trước: Bronze không được bắt đầu đọc tập file đang thay thế.
    manifest_key = DAY_PREFIX + "/manifest.json"
    if manifest_key in keys:
        response = s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects":[{"Key":manifest_key}]})
        if response.get("Errors"):
            raise RuntimeError("Không xóa được manifest ngày; dừng ghi Landing")
        keys.remove(manifest_key)
    for offset in range(0, len(keys), 1000):
        response = s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects":[{"Key":k} for k in keys[offset:offset+1000]]})
        if response.get("Errors"):
            raise RuntimeError("Xóa file Landing ngày không thành công; không chốt manifest")
    logger.info("Đã xóa tập file cũ tại s3://%s/%s", BUCKET_NAME, expected)

def fetch_page(session, url, headers, body):
    for attempt in range(4):
        try:
            response = session.post(url, headers=headers, json=body,
                                    timeout=(15, 120), verify=ca, allow_redirects=False)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == 3:
                raise RuntimeError("Kết nối API thất bại sau retry") from None
            time.sleep(2 ** attempt)
            continue
        if response.status_code in (429, 500, 502, 503, 504) and attempt < 3:
            time.sleep(2 ** attempt)
            continue
        logger.info("API HTTP status=%s", response.status_code)
        if response.status_code != 200:
            raise RuntimeError(f"API HTTP {response.status_code}; không ghi response body vào log")
        try:
            payload = response.json()
        except ValueError:
            raise ValueError("API không trả JSON") from None
        if not isinstance(payload, dict) or payload.get("success") is not True:
            raise ValueError("API success không bằng true")
        if not isinstance(payload.get("result"), list):
            raise ValueError("API result không phải danh sách")
        return payload
    raise RuntimeError("Không lấy được trang API")

def extract():
    validate_contract()
    end_mode = "next_day_boundary"
    base = required("HRM_BASE_URL").rstrip("/")
    parsed = urlparse(base)
    if parsed.scheme != "https" or not parsed.netloc or parsed.username or parsed.query or parsed.fragment or parsed.path not in ("", "/"):
        raise ValueError("HRM_BASE_URL phải là HTTPS origin")
    token = required("HRM_TOKEN")
    headers = {"Content-Type": "application/json", "Authorization": "Bearer " + token}
    body = {"access_token": token, "date_start": ETL_Date,
            "date_end": (BATCH_DATE+timedelta(days=1)).isoformat()}
    endpoint = "danhsachdonvi" if ENTITY == "unit" else "danhsachnhansu"
    logger.info("API window date_start=%s date_end=%s", body["date_start"], body["date_end"])
    page_param = size_param = None
    size = int(os.getenv("HRM_PAGE_SIZE", "1000"))
    max_pages = int(os.getenv("HRM_MAX_PAGES", "10000"))
    if size <= 0 or max_pages <= 0:
        raise ValueError("Page size/max pages phải dương")
    if ENTITY == "employee":
        page_param, size_param = os.getenv("HRM_PAGE_PARAM", "").strip(), os.getenv("HRM_SIZE_PARAM", "").strip()
        if bool(page_param) != bool(size_param):
            raise ValueError("Cần cấu hình cả HRM_PAGE_PARAM và HRM_SIZE_PARAM hoặc để cả hai trống")
        if page_param and (page_param == size_param or {page_param, size_param} & set(body)):
            raise ValueError("Tên tham số phân trang bị trùng")
    files, pending, seen, count, expected, fingerprints = [], [], {}, 0, None, set()
    with requests.Session() as session:
        for page in range(1, max_pages+1):
            if page_param:
                body.update({page_param: page, size_param: size})
            payload = fetch_page(session, base+"/api/nhansu/"+endpoint, headers, body)
            rows = payload["result"]
            logger.info("API page=%s success=%s rows=%s total=%s", page, payload.get("success"), len(rows), payload.get("total"))
            if ENTITY == "employee":
                for k in ("total", "pageindex", "pagesize"):
                    if type(payload.get(k)) is not int:
                        raise ValueError("Metadata phân trang phải là integer: " + k)
                if payload["pageindex"] != page or payload["pagesize"] <= 0 or payload["total"] < 0:
                    raise ValueError("Metadata phân trang không hợp lệ")
                if len(rows) > payload["pagesize"]:
                    raise ValueError("Số dòng trang vượt pagesize")
                if expected is None:
                    expected = payload["total"]
                elif expected != payload["total"]:
                    raise ValueError("Total thay đổi giữa các trang; chưa có tập dữ liệu ổn định")
            fp = hashlib.sha256(pack(rows)).hexdigest()
            if rows and fp in fingerprints:
                raise ValueError("API trả lặp trang")
            fingerprints.add(fp)
            for row in rows:
                if not isinstance(row, dict) or row.get("id") is None:
                    raise ValueError("Bản ghi thiếu id")
                key = str(row["id"])
                if key in seen:
                    raise ValueError("Trùng id giữa/trong các trang; cần đối soát nguồn")
                seen[key] = True
            key = f"{prefix(RUN_ID)}/page-{page:06d}.json"
            pending.append((key, payload, len(rows)))
            count += len(rows)
            if ENTITY == "unit" or count == expected:
                break
            if count > expected or not rows:
                raise ValueError("Thiếu/thừa dữ liệu so với total")
            if ENTITY == "employee" and not page_param:
                raise ValueError(f"API Employee gọi thành công nhưng mới nhận {count}/{expected} dòng. Cần tên tham số request phân trang trong HRM_PAGE_PARAM và HRM_SIZE_PARAM để lấy tiếp; chưa chốt hoặc thay thế Landing ngày.")
        else:
            raise ValueError("Vượt max pages; không chốt Landing")
    if count == 0 and not ALLOW_EMPTY:
        raise ValueError("Response rỗng chưa được cho phép")
    # Lấy và kiểm tra xong API rồi mới xóa dữ liệu cũ của đúng ngày.
    # Chỉ ghi manifest sau khi toàn bộ file mới đã ghi thành công.
    clear_daily_landing()
    for key, payload, row_count in pending:
        files.append({"key": key, "sha256": put_json(key, payload), "count": row_count})
    manifest = {"entity": ENTITY, "data_date": ETL_Date, "run_id": RUN_ID,
                "count": count, "files": files, "ingest_time": datetime.now(timezone.utc).isoformat(),
                "date_start": body["date_start"], "date_end": body["date_end"], "date_end_mode": end_mode, "source": SOURCE, "complete": True, "manifest_version": 3}
    put_json(f"{prefix(RUN_ID)}/manifest.json", manifest)
    return manifest



In [ ]:
globals().pop('LANDING_MANIFEST', None)
RUN_ID = uuid.uuid4().hex
status_key = f"{LOG_PREFIX}/landing-{RUN_ID}.json"
result = {"layer": "landing", "entity": ENTITY, "data_date": ETL_Date,
          "run_id": RUN_ID, "started_at": datetime.now(timezone.utc).isoformat()}
s3 = None
logger.info("START API -> Landing entity=%s date=%s run_id=%s", ENTITY, ETL_Date, RUN_ID)
try:
    validate_contract()
    s3 = init_landing()
    manifest = extract()
    path = f"s3://{BUCKET_NAME}/{prefix(RUN_ID)}/manifest.json"
    status = "Success"
    result.update(status="Success", record_count=manifest["count"], manifest_path=path)
    logger.info("SUCCESS rows=%s pages=%s", manifest["count"], len(manifest["files"]))
    print("LANDING_RUN_ID =", RUN_ID, flush=True)
    print("MANIFEST_PATH =", path, flush=True)
    LANDING_MANIFEST = manifest
except Exception as exc:
    status = "Fail"
    result.update(status="Fail", error_type=type(exc).__name__, error=error_summary(exc))
    logger.error("FAIL %s", result["error"])
    raise RuntimeError(result["error"]) from None
finally:
    log_message = result.get("error")
    record_count = result.get("record_count", 0)
    result["ended_at"] = datetime.now(timezone.utc).isoformat()
    if s3 is not None:
        try:
            put_json(status_key, result)
            logger.info("S3 log: s3://%s/%s", BUCKET_NAME, status_key)
        except Exception as log_exc:
            logger.error("Không ghi được log S3: %s", type(log_exc).__name__)



In [ ]:
# Kiểm tra manifest đã lưu; không gọi lại API.
if "LANDING_MANIFEST" not in globals():
    print("Chưa có lần chạy Landing thành công trong kernel này")
else:
    saved = get_json(f"{prefix(LANDING_MANIFEST['run_id'])}/manifest.json")
    print(json.dumps({k: saved[k] for k in ("entity", "data_date", "run_id", "count", "complete")}, indent=2))
    print("Số file:", len(saved["files"]))


## Vận hành
1. Import bằng Local file. Landing dùng Python với requests/boto3; Bronze dùng Python with Spark với connector như mẫu.
2. Restart kernel và Run All để thay thế các hàm cũ còn trong kernel. Truyền cùng ETL_Date cho hai notebook.
3. API đơn vị dùng request mẫu với access_token/date_start/date_end; không cần confirm. API nhân sự gọi trang đầu không cần cấu hình phân trang; nếu response còn trang tiếp theo thì cần tên tham số request thực tế trong HRM_PAGE_PARAM và HRM_SIZE_PARAM để hoàn tất. Kiểm tra total/số trang bằng response thực tế, không dùng cờ xác nhận.
4. Landing lấy và kiểm tra xong API; xóa tập file cũ đúng `hrm/<entity>/YYYYMMDD/`; ghi trang mới và ghi manifest cuối cùng. Nếu API lỗi trước bước xóa, tập cũ được giữ. Nếu lỗi trong lúc ghi file, không có manifest hoàn tất.
5. Bronze tự đọc `manifest.json` theo entity/ETL_Date; không cần copy đường dẫn. Để MANIFEST_PATH_CONFIG và LANDING_RUN_ID_CONFIG trống nếu muốn đọc tập hiện hành của ngày. Nếu truyền tùy chọn này, giá trị phải khớp manifest hiện tại.
6. Bronze overwrite atomic partition data_date=ETL_Date, tương đương thay thế toàn bộ dữ liệu ngày mà không có khoảng DELETE/APPEND. API success với 0 bản ghi vẫn thay thế ngày bằng tập rỗng; request lỗi không được coi là rỗng.
7. Chạy Landing xong mới chạy Bronze; không chạy lại Landing cùng ngày khi Bronze đang đọc. S3 không có atomic replace toàn thư mục; checksum phát hiện nội dung thay đổi giữa lúc đọc. Khi Landing đã chạy lại, replay sẽ dùng tập ngày mới, không còn tập run cũ.
8. Log Landing/backup log Bronze nằm ngoài dữ liệu ngày: `hrm/<entity>/_logs/YYYYMMDD/`. Log Bronze giữ schema bronze.pipeline.log_pipeline như mẫu. Cell cuối kiểm tra số lượng/manifest, không in dữ liệu nhân sự.
Giới hạn: tập dữ liệu ngày được giữ trong RAM driver; chưa khắc phục lỗi metastore của runtime. Các trường hợp total thay đổi hoặc lặp trang/bản ghi vẫn báo lỗi để tránh chốt dữ liệu thiếu/trùng.


## Cách gọi API đã kiểm chứng
POST JSON với cả `Authorization: Bearer <token>` và `access_token` trong body. Request này không phụ thuộc HRM_AUTH_MODE cũ trong kernel. Date_start=ETL_Date, date_end=ETL_Date+1 ngày. Employee có thể chỉ trả trang đầu: notebook không chốt tập thiếu trang. Restart kernel và Run All sau khi import bản mới. Hai notebook Bronze không thay đổi.